# Retailrocket Dataset Download

This notebook is designed to run in Google Colab and uses `data.py` to download a Kaggle dataset.

In [5]:
import sys
import csv
from pathlib import Path

project_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from data import download_kaggle_dataset, preprocess_retailrocket_data

Download the Retailrocket dataset into the local `data/` folder so the preprocessing step can work with the raw CSV files.

In [6]:
dataset_name = "retailrocket/ecommerce-dataset"
output_path = download_kaggle_dataset(dataset_name, output_dir=project_root / "data")
print(f"Downloaded dataset into: {output_path}")
sorted(str(path.relative_to(output_path)) for path in output_path.rglob("*") if path.is_file())[:20]

Downloaded dataset into: /home/mrjoezhong/serko-ds-test/data


['.complete/datasets/retailrocket/ecommerce-dataset/2/bundle.complete',
 'cate_properties.csv',
 'cate_properties_bucket_idx.csv',
 'category_lookup.csv',
 'category_tree.csv',
 'events.csv',
 'events_time_range.csv',
 'item_id_map.csv',
 'item_properties.csv',
 'item_properties_part1.csv',
 'item_properties_part2.csv',
 'item_property_key_map.csv',
 'non_numeric_properties.csv',
 'non_numeric_properties_bucket_idx.csv',
 'numeric_properties.csv',
 'numeric_properties_bucket_idx.csv',
 'property_id_map.csv',
 'property_mean_std.csv',
 'user_events.csv',
 'user_item.csv']

Run the shared preprocessing helper once to prepare the derived Retailrocket tables.

`preprocess_retailrocket_data(...)` currently does four things:
- builds `category_lookup.csv` from `category_tree.csv`
- computes the category leaf-depth summary returned as `leaf_depth_statistics`
- merges `item_properties_part1.csv` and `item_properties_part2.csv` into `item_properties.csv`
- splits the merged item properties into 3 different item - property files.
- generate user_events and user_item files.
- The following should take 5 minutes and only needs to run once.

In [3]:
preprocess_outputs = preprocess_retailrocket_data(output_path)
category_lookup_path = preprocess_outputs["category_lookup_path"]
leaf_depth_stats = preprocess_outputs["leaf_depth_statistics"]
merged_item_properties_path = preprocess_outputs["merged_item_properties_path"]

print(f"Generated category lookup table at: {category_lookup_path}")
print(f"Merged item properties file at: {merged_item_properties_path}")
print("Leaf depth statistics:")
for depth, count in leaf_depth_stats.items():
    print(f"depth={depth}, count={count}")

with category_lookup_path.open(newline="", encoding="utf-8") as handle:
    preview_rows = list(csv.reader(handle))[:10]

preview_rows

Finished generate_category_lookup_table -> /home/mrjoezhong/serko-ds-test/data/category_lookup.csv
Finished merge_item_properties_files -> /home/mrjoezhong/serko-ds-test/data/item_properties.csv


Loading item property rows: 20275902row [01:29, 227255.83row/s]
Deduplicating item-property groups: 100%|██████████| 10124706/10124706 [00:20<00:00, 503988.27group/s]


Finished load_item_property_row_groups -> numeric=1,440,359, category=1,031,473, non_numeric=10,461,690
Finished process_numeric_property_values -> /home/mrjoezhong/serko-ds-test/data/numeric_properties.csv
Finished process_category_property_values -> /home/mrjoezhong/serko-ds-test/data/cate_properties.csv
Finished process_non_numeric_property_values -> /home/mrjoezhong/serko-ds-test/data/non_numeric_properties.csv


Scanning event timestamps: 2756101row [00:05, 489635.02row/s]


Finished generate_events_time_range -> /home/mrjoezhong/serko-ds-test/data/events_time_range.csv


Scanning event timestamps: 2756101row [00:05, 481721.15row/s]
Grouping events by visitor-item: 2756101row [00:06, 456414.66row/s]
Writing user event lists: 100%|██████████| 1407580/1407580 [00:02<00:00, 534879.73user/s]


Finished merge_events_by_visitor_item -> /home/mrjoezhong/serko-ds-test/data/user_item.csv and /home/mrjoezhong/serko-ds-test/data/user_events.csv (negative=1,931,118, positive=214,061, negative_cold_start=87, positive_cold_start=13,595)
Finished write_bucket_index_property_files -> /home/mrjoezhong/serko-ds-test/data/numeric_properties_bucket_idx.csv, /home/mrjoezhong/serko-ds-test/data/cate_properties_bucket_idx.csv, and /home/mrjoezhong/serko-ds-test/data/non_numeric_properties_bucket_idx.csv
Generated category lookup table at: /home/mrjoezhong/serko-ds-test/data/category_lookup.csv
Merged item properties file at: /home/mrjoezhong/serko-ds-test/data/item_properties.csv
Leaf depth statistics:
depth=1, count=1
depth=2, count=50
depth=3, count=525
depth=4, count=632
depth=5, count=86
depth=6, count=13


[['level_1_category_id',
  'level_2_category_id',
  'level_3_category_id',
  'level_4_category_id',
  'level_5_category_id',
  'level_6_category_id'],
 ['140', '61', '323', '1558', '', ''],
 ['140', '61', '323', '', '', ''],
 ['140', '61', '897', '120', '', ''],
 ['140', '61', '897', '1098', '', ''],
 ['140', '61', '897', '1317', '', ''],
 ['140', '61', '897', '1540', '', ''],
 ['140', '61', '897', '1588', '', ''],
 ['140', '61', '897', '', '', ''],
 ['140', '61', '1003', '', '', '']]

Build or load the saved all-item embedding table produced by `model.get_all_items_embedding(...)`.

Load the iterable user-item dataset from `user_item.csv` and split it into `train_set` and `test_set` with the dataset's built-in `split(...)` helper.

The following data loading would take 3 minutes to load all the data.

In [7]:
from pathlib import Path

from data import UserItemEmbeddingIterableDataset
from model import initialize_item_embedding_resources

user_item_path = Path(output_path) / "user_item.csv"

state = initialize_item_embedding_resources(dataset_path=output_path)

positive_negative_ratio = 1.0

dataset = UserItemEmbeddingIterableDataset(
    user_item_path=user_item_path,
    resources=state.resources,
    token_transformer=state.token_transformer,
    item_projection=state.item_projection,
    user_projection=state.user_projection,
    available_filter=1,
    positive_negative_ratio=positive_negative_ratio,
    device=state.device,
)

train_set, test_set = dataset.split(train_size=0.8, seed=42)

print(f"train rows: {len(train_set)}")
print(f"test rows: {len(test_set)}")


train rows: 700306
test rows: 175077


In [8]:
from torch.optim import Adam
from model import FactorizationMachines

learning_rate = 1e-3

fm_model = FactorizationMachines(
    embedding_dim=state.item_projection.out_features,
    latent_dim=16,
).to(state.device)
optimizer = Adam(fm_model.parameters(), lr=learning_rate)


In [9]:
%env WANDB_API_KEY=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

env: WANDB_API_KEY=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


In [ ]:
import torch
from tqdm.auto import tqdm
import wandb
from torch.utils.data import DataLoader
from data import collate_user_item_embedding_batch


batch_size = 64
epochs = 3
wandb_project = "serko-ds-test"

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    collate_fn=collate_user_item_embedding_batch,
)
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    collate_fn=collate_user_item_embedding_batch,
)

wandb_run = wandb.init(
    project=wandb_project,
    config={
        "batch_size": batch_size,
        "epochs": epochs,
        "learning_rate": learning_rate,
        "fm_latent_dim": fm_model.latent_dim,
        "embedding_dim": fm_model.embedding_dim,
    },
)

try:
    global_step = 0
    for epoch in range(epochs):
        fm_model.train()
        train_loss_total = 0.0
        train_correct = 0
        train_example_count = 0

        train_progress = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [train]")
        for batch in train_progress:
            optimizer.zero_grad()
            logits, loss = fm_model.forward_batch(batch)
            loss.backward()
            optimizer.step()

            labels = batch[2]
            predictions = (torch.sigmoid(logits) >= 0.5).to(labels.dtype)
            batch_size_actual = labels.numel()
            train_loss_total += loss.item() * batch_size_actual
            train_correct += (predictions == labels).sum().item()
            train_example_count += batch_size_actual

            avg_train_loss_so_far = train_loss_total / max(train_example_count, 1)
            avg_train_accuracy_so_far = train_correct / max(train_example_count, 1)
            global_step += 1
            wandb.log(
                {
                    "train/step_loss": loss.item(),
                    "train/step_accuracy": (predictions == labels).float().mean().item(),
                    "train/running_loss": avg_train_loss_so_far,
                    "train/running_accuracy": avg_train_accuracy_so_far,
                    "train/epoch": epoch + 1,
                    "trainer/global_step": global_step,
                    "trainer/examples_seen": train_example_count,
                },
                step=global_step,
            )
            train_progress.set_postfix(
                loss=f"{avg_train_loss_so_far:.4f}",
                accuracy=f"{avg_train_accuracy_so_far:.4f}",
            )

        fm_model.eval()
        test_loss_total = 0.0
        test_correct = 0
        test_example_count = 0
        test_true_positive = 0
        test_false_negative = 0
        test_false_positive = 0

        with torch.no_grad():
            test_progress = tqdm(test_loader, desc=f"Epoch {epoch + 1}/{epochs} [test]")
            for batch in test_progress:
                logits, loss = fm_model.forward_batch(batch)
                labels = batch[2]
                predictions = (torch.sigmoid(logits) >= 0.5).to(labels.dtype)
                batch_size_actual = labels.numel()
                test_loss_total += loss.item() * batch_size_actual
                test_correct += (predictions == labels).sum().item()
                test_example_count += batch_size_actual
                test_true_positive += ((predictions == 1) & (labels == 1)).sum().item()
                test_false_negative += ((predictions == 0) & (labels == 1)).sum().item()
                test_false_positive += ((predictions == 1) & (labels == 0)).sum().item()

                avg_test_loss_so_far = test_loss_total / max(test_example_count, 1)
                avg_test_accuracy_so_far = test_correct / max(test_example_count, 1)
                test_progress.set_postfix(
                    loss=f"{avg_test_loss_so_far:.4f}",
                    accuracy=f"{avg_test_accuracy_so_far:.4f}",
                )

        avg_train_loss = train_loss_total / max(train_example_count, 1)
        avg_train_accuracy = train_correct / max(train_example_count, 1)
        avg_test_loss = test_loss_total / max(test_example_count, 1)
        avg_test_accuracy = test_correct / max(test_example_count, 1)
        test_recall = test_true_positive / max(test_true_positive + test_false_negative, 1)
        test_precision = test_true_positive / max(test_true_positive + test_false_positive, 1)
        test_f1 = 0.0 if (test_precision + test_recall) == 0 else 2 * test_precision * test_recall / (test_precision + test_recall)

        wandb.log(
            {
                "epoch": epoch + 1,
                "train/loss": avg_train_loss,
                "train/accuracy": avg_train_accuracy,
                "test/loss": avg_test_loss,
                "test/accuracy": avg_test_accuracy,
                "test/recall": test_recall,
                "test/f1": test_f1,
                "trainer/global_step": global_step,
            }
            ,
            step=global_step,
        )
        print(
            f"epoch {epoch + 1}: train_loss={avg_train_loss:.4f}, train_accuracy={avg_train_accuracy:.4f}, "
            f"test_loss={avg_test_loss:.4f}, test_accuracy={avg_test_accuracy:.4f}, test_recall={test_recall:.4f}, test_f1={test_f1:.4f}"
        )
finally:
    wandb.finish()


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: mrjoezhong (mrjoezhong-research) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/3 [train]:   0%|          | 6/10943 [05:14<154:36:37, 50.89s/it, accuracy=0.5130, loss=39368514167142023168.0000] 